<a href="https://colab.research.google.com/github/JuanZapa7a/AINavalEngineering/blob/main/NB08_Decision_Trees_Ensembles_and_SVM_with_Real_Sonar_Data_ES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# **NB08 · Clase 8 — Árboles de decisión, ensembles y SVM con datos reales de sonar**

## Bloque 2: IA — Machine Learning (continuación)

`NB07` entrenó un clasificador (regresión logística) y un par de modelos de regresión (regresión lineal frente a Random Forest). Esta clase profundiza en el lado de la clasificación: cómo dividen realmente los datos los árboles de decisión, cómo combinan los ensembles (bagging y boosting) muchos modelos débiles en uno fuerte, y cómo encuentran las máquinas de vectores de soporte (SVM) un límite de separación — y después los compara a todos, de forma rigurosa, sobre un dataset real: el clásico dataset **[Sonar, Mines vs. Rocks](https://archive.ics.uci.edu/dataset/151/connectionist+bench+sonar+mines+vs+rocks)** (208 ecos reales de sonar, 60 características de bandas de frecuencia, de origen genuinamente naval/submarino), ya reflejado en este repositorio en [`Datasets/sonar.all-data`](https://github.com/JuanZapa7a/AINavalEngineering/blob/main/Datasets/sonar.all-data).

### Objetivos de aprendizaje

Al terminar esta clase, el alumnado será capaz de:
- Explicar cómo divide los datos un árbol de decisión (impureza de Gini) y por qué los árboles sin restricciones sobreajustan.
- Distinguir el bagging (Random Forest) del boosting (AdaBoost), y explicar por qué los ensembles suelen superar a un único árbol.
- Explicar la idea central de una máquina de vectores de soporte: maximizar el margen, los vectores de soporte y el truco del kernel.
- Construir un `Pipeline` seguro frente a fugas que escale los datos *dentro* de la validación cruzada, no antes.
- Comparar de forma justa varios clasificadores con validación cruzada, y leer las importancias de características de un Random Forest.
- Ejecutar una primera búsqueda de hiperparámetros con `GridSearchCV`.

### Agenda (clase de 2 horas)

| # | Segmento de la clase | Duración aprox. | Tipo |
|---|---------------------|:---:|:---:|
| 1 | Repaso de NB01–NB07, hoja de ruta de hoy | 5 min | Teoría |
| 2 | Carga y exploración del dataset real (Sonar: minas vs. rocas) | 15 min | Práctica |
| 3 | Árboles de decisión: cómo funciona CART, y una demo práctica de sobreajuste | 15 min | Teoría + Práctica |
| 4 | Ensembles: bagging (Random Forest) frente a boosting (AdaBoost) | 15 min | Teoría |
| 5 | Máquinas de vectores de soporte: margen, vectores de soporte, kernels | 10 min | Teoría |
| 6 | Preprocesado y pipelines seguros frente a fugas | 15 min | Teoría + Práctica |
| 7 | Comparación de cuatro clasificadores con validación cruzada | 10 min | Práctica |
| 8 | Importancia de características de Random Forest | 15 min | Práctica |
| 9 | Una primera búsqueda de hiperparámetros con `GridSearchCV` | 15 min | Teoría + Práctica |
| 10 | Resumen, tarea, próxima clase | 5 min | Teoría |

> Los tiempos son orientación aproximada, no un guion cerrado — no hay descansos programados. Si se cubre todo con tiempo de sobra, la clase termina antes; eso puede pasar y no pasa nada.

---

## 1. Repaso: dónde estamos

- **`NB01`**: historia de la IA, por qué importa la IA para la ingeniería naval/oceánica.
- **`NB02`**: fundamentos de Python, Colab, NumPy, primer dataset de Pandas.
- **`NB07`**: el flujo de ML de principio a fin — train/val/test, sobreajuste, métricas, un clasificador y un regresor, validación cruzada, y un ejemplo real de fuga de datos.
- **`NB08`** (hoy): algoritmos de clasificación concretos, en profundidad, comparados todos de forma justa sobre el mismo dataset real.

Seguimos dentro del **Bloque 2 — IA: Machine Learning** de la hoja de ruta del curso.

---

## 2. Carga y exploración del dataset real

El dataset **[Sonar, Mines vs. Rocks](https://archive.ics.uci.edu/dataset/151/connectionist+bench+sonar+mines+vs+rocks)** se recopiló rebotando señales de sonar sobre un cilindro metálico (simulando una mina) y sobre rocas, con distintos ángulos. Cada uno de los 208 registros es un eco de sonar, representado por 60 números (energía en 60 bandas de frecuencia, cada una en el rango 0.0–1.0) — un problema genuinamente naval de acústica submarina: distinguir un objeto similar a una mina de una roca usando solo el sonar.

In [ ]:
!wget -q -O sonar.csv https://raw.githubusercontent.com/JuanZapa7a/AINavalEngineering/main/Datasets/sonar.all-data

import pandas as pd

sonar = pd.read_csv("sonar.csv", header=None)
sonar.columns = [f"freq_{i}" for i in range(60)] + ["label"]
print(sonar.shape)
sonar.head()

`label` es `M` (mina) o `R` (roca). Comprobemos el balance de clases y confirmemos que no hay valores ausentes antes de hacer nada más — exactamente las mismas primeras preguntas que hicimos para cada dataset en `NB02`/`NB07`.

In [ ]:
print(sonar["label"].value_counts())
print()
print("missing values:", sonar.isna().sum().sum())

Construiremos nuestra matriz de características `X` (las 60 bandas de frecuencia) y un objetivo binario `y` (1 = mina, 0 = roca) una sola vez, y los reutilizaremos para todos los modelos de esta clase.

In [ ]:
X = sonar.drop(columns="label")
y = (sonar["label"] == "M").astype(int)

print("X shape:", X.shape, " mine ratio:", y.mean().round(3))

---

## 3. Árboles de decisión

Un **árbol de decisión** (CART — Classification and Regression Trees) predice haciendo una secuencia de preguntas de sí/no sobre las características, p. ej. "¿es `freq_11` > 0.18?". En cada paso, elige la división que mejor separa las clases, medida con la **impureza de Gini**:

$$
\text{Gini} = 1 - \sum_{k} p_k^2
$$

donde $p_k$ es la proporción de la clase $k$ en un nodo. Un nodo puro (todo una clase) tiene Gini = 0; una división 50/50 tiene la impureza máxima. El árbol sigue dividiendo, de forma voraz, para reducir la impureza — y, sin control, `seguirá dividiendo hasta clasificar perfectamente cada punto de entrenamiento`. Eso es **sobreajuste**, exactamente el concepto de `NB07`: un árbol que memoriza el conjunto de entrenamiento funciona mal con datos nuevos.

`max_depth` (y parámetros similares como `min_samples_leaf`) controlan esto: los árboles poco profundos infraajustan, los árboles sin restricciones sobreajustan. Veámoslo directamente, sobre nuestro dataset real.

> **Para saber más**: [Aprendizaje de árboles de decisión (Wikipedia)](https://en.wikipedia.org/wiki/Decision_tree_learning) · [documentación de `sklearn.tree.DecisionTreeClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html).

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

Entrena árboles de decisión con `max_depth` creciente, y registra la precisión tanto en el conjunto de entrenamiento como en el conjunto de test reservado:

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

depths = range(1, 15)
train_acc, test_acc = [], []
for d in depths:
    tree = DecisionTreeClassifier(max_depth=d, random_state=42)
    tree.fit(X_train, y_train)
    train_acc.append(accuracy_score(y_train, tree.predict(X_train)))
    test_acc.append(accuracy_score(y_test, tree.predict(X_test)))

plt.plot(depths, train_acc, marker="o", label="Train accuracy")
plt.plot(depths, test_acc, marker="o", label="Test accuracy")
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.title("Decision tree: overfitting as depth increases")
plt.legend()
plt.show()

**Interpreta tu propia gráfica**: la precisión de entrenamiento debería subir hacia 1.0 a medida que crece `max_depth` — el árbol siempre puede ajustar mejor sus propios datos de entrenamiento con más divisiones. La precisión de test se comporta de forma distinta: normalmente sube al principio, y luego se estanca o cae en cuanto el árbol es lo bastante profundo como para empezar a memorizar ruido. La brecha entre ambas curvas *es* el sobreajuste, hecho visible. ¿Dónde dejarías de hacer crecer el árbol?

**Pruébalo tú mismo**: lee la gráfica anterior y elige el `max_depth` donde la precisión de test alcanza su máximo (o se estabiliza) — después entrena un árbol exactamente con esa profundidad e imprime su `classification_report` completo sobre el conjunto de test.

In [ ]:
best_depth = test_acc.index(max(test_acc)) + 1   # depths start at 1
best_tree = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
best_tree.fit(X_train, y_train)

from sklearn.metrics import classification_report
print(f"Best max_depth from the plot: {best_depth}")
print(classification_report(y_test, best_tree.predict(X_test), target_names=["Rock", "Mine"]))


---

## 4. Ensembles: combinando muchos modelos

Un único árbol de decisión es un modelo de **varianza alta**: pequeños cambios en los datos de entrenamiento pueden producir un árbol muy distinto (probablemente hayas visto esa inestabilidad reflejada en lo irregular de la curva de precisión de test anterior). Los **métodos de ensemble** entrenan muchos modelos y combinan sus predicciones, lo que reduce la varianza sin aumentar necesariamente el sesgo — una aplicación directa y práctica del equilibrio sesgo-varianza de `NB07`.

### Bagging: Random Forest

El **bagging** (Bootstrap Aggregating) entrena muchos árboles, cada uno sobre una muestra bootstrap aleatoria de los datos de entrenamiento, y después promedia (o vota) sus predicciones. **Random Forest** añade una fuente más de aleatoriedad: en cada división, solo considera un subconjunto aleatorio de características, lo que descorrelaciona aún más los árboles — así sus errores tienen menos probabilidad de coincidir, y el promedio cancela más ruido.

### Boosting: AdaBoost

El **boosting** construye modelos *secuencialmente*: cada modelo nuevo se centra en los ejemplos de entrenamiento que los anteriores fallaron (reponderándolos), y después todos los modelos votan, ponderados según su precisión. **AdaBoost** (Adaptive Boosting) es el ejemplo clásico. Mientras que el bagging reduce la varianza promediando modelos independientes, el boosting reduce el sesgo encadenando modelos que corrigen los errores de los anteriores.

| Método | Modelos entrenados | Cómo se combinan | Reduce principalmente | Ejemplo |
|---|---|---|---|---|
| Bagging | De forma independiente, en paralelo, sobre muestras bootstrap | Promedio / voto | Varianza | Random Forest |
| Boosting | Secuencialmente, cada uno corrigiendo al anterior | Voto ponderado | Sesgo | AdaBoost |

Ambos aparecerán en nuestra comparación de más abajo, junto al árbol de decisión individual, para poder ver el efecto directamente en números reales de precisión y no solo en la teoría.

> **Para saber más**: [Bootstrap aggregating (Wikipedia)](https://en.wikipedia.org/wiki/Bootstrap_aggregating) · [Random forest (Wikipedia)](https://en.wikipedia.org/wiki/Random_forest) · [Boosting (Wikipedia)](https://en.wikipedia.org/wiki/Boosting_%28machine_learning%29) · [AdaBoost (Wikipedia)](https://en.wikipedia.org/wiki/AdaBoost) · [documentación de `sklearn.ensemble.RandomForestClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) · [documentación de `sklearn.ensemble.AdaBoostClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html).

---

## 5. Máquinas de vectores de soporte

Una **máquina de vectores de soporte (SVM)** busca el límite entre clases que maximiza el **margen** — la distancia a los puntos de entrenamiento más cercanos de cada clase. Esos puntos más cercanos son los **vectores de soporte**; son ellos, y solo ellos, los que determinan dónde queda el límite (los puntos lejos del límite no importan).

Los datos reales rara vez son linealmente separables, así que las SVM suelen usar el **truco del kernel**: en vez de transformar explícitamente las características a un espacio de mayor dimensión donde un límite recto funcionaría, una función kernel calcula el resultado equivalente directamente, sin llegar a formar nunca esa transformación costosa. El **kernel RBF (función de base radial)** es la opción habitual por defecto para problemas no lineales — es el que usaremos a continuación.

Las SVM son conocidas por funcionar bien en datasets con **muchas características y relativamente pocas muestras** — exactamente la forma de nuestro dataset Sonar (208 filas, 60 características) — por eso se incluye en esta comparación en vez de quedar como un apunte teórico.

> **Para saber más**: [Máquina de vectores de soporte (Wikipedia)](https://en.wikipedia.org/wiki/Support_vector_machine) · [Método kernel (Wikipedia)](https://en.wikipedia.org/wiki/Kernel_method) · [documentación de `sklearn.svm.SVC`](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html).

---

## 6. Preprocesado y pipelines seguros frente a fugas

Las SVM (y los métodos basados en distancias en general) son sensibles a la escala de las características, así que escalamos las 60 bandas de frecuencia con `StandardScaler`, igual que escalamos las características numéricas en `NB07`.

Hay un matiz con el que `NB07` no tuvo que lidiar: cuando hacemos validación cruzada (entrenar/evaluar muchas veces sobre distintos folds), el escalado **debe ajustarse solo con los datos de entrenamiento de cada fold**, no con el dataset completo de antemano — si no, `la información de los datos de test de cada fold se filtra a la media/varianza del escalador, inflando silenciosamente nuestra estimación de precisión`. Es la misma idea de *fuga de datos* de `NB07`, aplicada a un paso de preprocesado en vez de a una característica.

El `Pipeline` de `scikit-learn` resuelve esto automáticamente: agrupar `StandardScaler` y un modelo en un único objeto `Pipeline` hace que la validación cruzada reajuste el escalador desde cero en cada fold.

> **Para saber más**: [documentación de `sklearn.pipeline.Pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

example_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC(kernel="rbf", random_state=42)),
])
example_pipe

**Pruébalo tú mismo**: mide la fuga directamente. Escala `X` una vez sobre el dataset *completo* (la forma incorrecta), y después haz validación cruzada de una SVM sobre esos datos ya escalados — compara su precisión media con la versión honesta del `Pipeline`. ¿La puntuación con fuga es más alta, más baja, o parecida?

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv_leak_check = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scaler_leaky = StandardScaler()
X_scaled_leaky = scaler_leaky.fit_transform(X)   # WRONG: fit on the whole dataset before cross-validation
leaky_scores = cross_val_score(SVC(kernel="rbf", random_state=42), X_scaled_leaky, y, cv=cv_leak_check, scoring="accuracy")

honest_pipe = Pipeline([("scaler", StandardScaler()), ("model", SVC(kernel="rbf", random_state=42))])
honest_scores = cross_val_score(honest_pipe, X, y, cv=cv_leak_check, scoring="accuracy")

print(f"Leaky (scaled before CV) mean accuracy:  {leaky_scores.mean():.4f}")
print(f"Honest (Pipeline, scaled inside CV) mean accuracy: {honest_scores.mean():.4f}")


Con solo 208 filas, el efecto suele ser pequeño — pero en un dataset más grande y más sensible puede importar mucho más. La dirección (con fuga ≥ honesto) es la verdadera lección, no el tamaño exacto de la diferencia en este dataset concreto.

---

## 7. Comparación de cuatro clasificadores con validación cruzada

Ahora juntemos todo: **árbol de decisión**, **Random Forest** (bagging), **AdaBoost** (boosting) y **SVM (RBF)** — cada uno envuelto en el mismo `Pipeline` de escalado, evaluado con la misma validación cruzada estratificada de 5 folds, sobre los mismos datos reales. Este enfoque de "comprobación rápida de algoritmos" — probar varios algoritmos razonables en condiciones idénticas antes de comprometerse con uno — es práctica habitual: `casi nunca se sabe de antemano qué algoritmo funcionará mejor en un dataset nuevo`.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "SVM (RBF)": SVC(kernel="rbf", random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = {}
for name, model in models.items():
    pipe = Pipeline([("scaler", StandardScaler()), ("model", model)])
    cv_results[name] = cross_val_score(pipe, X, y, cv=cv, scoring="accuracy")

results_df = pd.DataFrame(cv_results)
results_df.mean().sort_values(ascending=False)

La tabla anterior muestra la precisión media; oculta lo *consistente* que es cada modelo entre folds. Un diagrama de cajas muestra ambas cosas a la vez:

In [ ]:
results_df.boxplot(figsize=(8, 5))
plt.ylabel("Cross-validated accuracy")
plt.title("Algorithm comparison — Sonar: Mines vs. Rocks")
plt.show()

**Interpreta tus propios resultados**: ¿es el árbol de decisión individual el más débil de los cuatro, y son tanto Random Forest como AdaBoost mejoras claras sobre él — coherente con el argumento sesgo-varianza de la Parte 4? ¿Se solapa la caja de la SVM con la de los ensembles, o queda claramente por delante/detrás? Un modelo con una *mediana más alta pero una caja más ancha* es más arriesgado que uno con una *mediana algo más baja pero una caja más estrecha* — ¿cuál usarías con más confianza para un sistema operativo de detección de minas, y por qué?

---

## 8. Importancia de características de Random Forest

A diferencia de una única predicción opaca, un Random Forest puede decirnos **en qué características se apoyó más**, promediado sobre todos sus árboles. Aquí, eso significa: `cuáles de las 60 bandas de frecuencia aportan más información para distinguir minas de rocas`.

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X, y)

importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
importances.head(10)

Representa las 15 principales para una lectura más rápida:

In [ ]:
importances.head(15).plot(kind="barh", figsize=(6, 6))
plt.gca().invert_yaxis()
plt.xlabel("Importance")
plt.title("Top 15 most informative frequency bands (Random Forest)")
plt.show()

Si un puñado pequeño de bandas de frecuencia domina, eso es una pista de que un modelo más simple usando solo esas bandas podría rendir casi igual de bien — útil si alguna vez necesitas reducir el número de canales de sensor o el cómputo en hardware real.

**Pruébalo tú mismo**: el `feature_importances_` incorporado (basado en Gini) se calcula a partir de las divisiones del conjunto de entrenamiento, así que puede estar sesgado hacia las características que el modelo usó al construir el árbol. `sklearn.inspection.permutation_importance` mide algo más directo: cuánto cae realmente la precisión en datos reservados cuando se mezcla cada característica — por eso debe calcularse sobre un conjunto de test genuino, no sobre los datos de entrenamiento (pruébalo sobre los datos de entrenamiento y obtendrías todo ceros: las predicciones de un modelo sobreajustado apenas cambian cuando mezclas una característica que ya ha memorizado). Compara sus 10 principales con las 10 principales basadas en Gini — ¿cuánto coinciden realmente?

In [ ]:
from sklearn.inspection import permutation_importance

rf_for_perm = RandomForestClassifier(n_estimators=200, random_state=42)
rf_for_perm.fit(X_train, y_train)

perm_result = permutation_importance(rf_for_perm, X_test, y_test, n_repeats=10, random_state=42)
perm_importances = pd.Series(perm_result.importances_mean, index=X.columns).sort_values(ascending=False)

gini_importances_train = pd.Series(rf_for_perm.feature_importances_, index=X.columns).sort_values(ascending=False)

print("Top 10 by Gini importance (same train/test split):", list(gini_importances_train.head(10).index))
print("Top 10 by permutation importance (held-out test set):", list(perm_importances.head(10).index))
print("How many overlap?", len(set(gini_importances_train.head(10).index) & set(perm_importances.head(10).index)))


---

## 9. Una primera búsqueda de hiperparámetros

Todos los modelos anteriores usaron su configuración por defecto. `GridSearchCV` `automatiza probar combinaciones de hiperparámetros, usando validación cruzada para puntuar cada una de forma justa, y se queda con la mejor`. Para la SVM, los dos hiperparámetros que más importan con un kernel RBF son `C` (cuánto penalizar los puntos mal clasificados — menor significa un margen más ancho y blando) y `gamma` (hasta dónde llega la influencia de un único punto de entrenamiento — menor significa límites más suaves).

> **Para saber más**: [Optimización de hiperparámetros (Wikipedia)](https://en.wikipedia.org/wiki/Hyperparameter_optimization) · [documentación de `sklearn.model_selection.GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html).

In [ ]:
from sklearn.model_selection import GridSearchCV

svm_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC(kernel="rbf", random_state=42)),
])

param_grid = {
    "model__C": [0.1, 1, 10, 100],
    "model__gamma": ["scale", 0.01, 0.1, 1],
}

grid = GridSearchCV(svm_pipe, param_grid, cv=cv, scoring="accuracy")
grid.fit(X, y)

print("Best params:", grid.best_params_)
print("Best CV accuracy:", round(grid.best_score_, 3))

**Pruébalo tú mismo**: `grid.cv_results_` guarda la puntuación de cada combinación, no solo la mejor. Reorganiza `mean_test_score` en una rejilla `C` x `gamma` y visualízala como un mapa de calor — ¿está el óptimo en una región amplia y estable, o en un pico estrecho rodeado de puntuaciones mucho peores?

In [ ]:
scores_matrix = grid.cv_results_["mean_test_score"].reshape(
    len(param_grid["model__C"]), len(param_grid["model__gamma"])
)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(scores_matrix, cmap="viridis")
ax.set_xticks(range(len(param_grid["model__gamma"])))
ax.set_xticklabels(param_grid["model__gamma"])
ax.set_yticks(range(len(param_grid["model__C"])))
ax.set_yticklabels(param_grid["model__C"])
ax.set_xlabel("gamma")
ax.set_ylabel("C")
ax.set_title("Grid search: mean CV accuracy")
plt.colorbar(im, ax=ax, label="Accuracy")
plt.show()


Compara `grid.best_score_` con el resultado sencillo de la `SVM (RBF)` de la Parte 7 — ajustar `C` y `gamma` debería igualar o superar la configuración por defecto, ya que la búsqueda en rejilla incluye el valor por defecto (`C=1, gamma="scale"`) como uno de sus candidatos.

---

## Resumen de la clase

- Los árboles de decisión dividen según la impureza de Gini y sobreajustan sin un límite de profundidad — vimos crecer directamente la brecha de precisión train/test, sobre datos reales.
- Los ensembles reducen el error de dos formas distintas: el bagging (Random Forest) reduce la varianza promediando árboles independientes; el boosting (AdaBoost) reduce el sesgo encadenando modelos que corrigen los errores de los anteriores.
- Las SVM maximizan el margen entre clases y usan kernels para manejar límites no lineales; se adaptan bien a datasets con muchas características y pocas muestras, como el nuestro.
- El escalado debe ocurrir *dentro* de la validación cruzada, no antes — `Pipeline` lo hace automático y evita una forma sutil de fuga de datos.
- Comparamos de forma justa cuatro clasificadores con validación cruzada, leímos las importancias de características de un Random Forest, y ejecutamos una primera `GridSearchCV`.

## Para la próxima clase (NB09)

Pasaremos del aprendizaje supervisado al **aprendizaje no supervisado**: clustering (K-Means) y reducción de dimensionalidad (PCA) — agrupar buques o travesías por similitud sin ninguna etiqueta.

## Tarea / Ideas de práctica

1. Añade k-vecinos más cercanos (k-NN) y Naive Bayes a la comparación de la Parte 7 — ¿cómo se sitúan frente a los cuatro modelos que probamos?
2. Repite la demo de sobreajuste de la Parte 3, pero para `RandomForestClassifier` variando `n_estimators` en vez de `DecisionTreeClassifier` variando `max_depth` — ¿llega a perjudicar la precisión de test tener más árboles?
3. Amplía la búsqueda en rejilla de la Parte 9 para incluir `kernel="linear"` como opción — ¿rinde notablemente peor un kernel lineal que el RBF en este dataset? ¿Qué te diría eso sobre los datos?
4. Usando las importancias de características de la Parte 8, reentrena un Random Forest usando solo las 10 bandas de frecuencia principales. ¿Cuánta precisión (si alguna) se pierde al descartar las otras 50 características?
5. Con tus propias palabras, explica por qué ajustar `StandardScaler` sobre el dataset completo antes de la validación cruzada sería una forma de fuga de datos — y cuál sería la consecuencia práctica (no solo teórica).

> ***Como siempre: enmarca cada pregunta en torno a lo que realmente necesitaría un sistema operativo de sonar/detección de minas — no solo "qué número es más alto".***

> Para un tratamiento más profundo, a nivel de libro de texto, de árboles, ensembles y SVM en un solo lugar, consulta el libro de texto gratuito [*An Introduction to Statistical Learning*](https://www.statlearning.com/) (James, Witten, Hastie & Tibshirani).